# EV Charging Hourly Demand Prediction

**AI Services MS2 | Advanced Techniques Supplementary G | 60 Marks | Exactly 20 Tests**

**CANDIDATE NOTEBOOK — complete all TODO function bodies.**

Input CSV files are in this folder. Generated model and report files are created only when workflow functions run.

## 1. Imports, parameters and feature groups


In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error

TARGET = "session_count"
RAW_NUMERIC = ["temperature_c", "feels_like_c", "humidity_pct", "wind_speed_kmph", "hour"]
NUMERIC_FEATURES = [
    "temperature_c", "feels_like_c", "humidity_pct", "wind_speed_kmph",
    "month", "is_weekend", "hour_sin", "hour_cos"
]
CATEGORICAL = ["season", "weather_condition", "holiday", "working_day"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL


## 2. Independently testable functions

Complete `load_data`, `build_pipeline`, `prepare_datasets`, `train_models`, `evaluate_models`, `select_best_model` and `predict_demand`. Follow exact return types in the instructions.

In [ ]:
def load_data(filename: str) -> pd.DataFrame:
    """Clean hourly records and engineer time features."""

    df = pd.read_csv(filename,dtype={'observation_id':'string'})
    df = df.drop_duplicates(subset=['observation_id'],keep='first')

    df['observation_date'] = pd.to_datetime(df['observation_date'],format='mixed',dayfirst=True,errors='coerce')
    df = df.dropna(subset=['observation_date'])

    for col in RAW_NUMERIC:
        df[col] = pd.to_numeric(df[col],errors='coerce').replace([np.inf,-np.inf],np.nan)

    for col in CATEGORICAL:
        df[col] = df[col].astype('string').str.strip().str.lower().replace(r'(^[+-]?\d+)(\.0+$)',r'\1',regex=True).replace('',pd.NA).fillna('unknown')

    df['month'] = df['observation_date'].dt.month

    df['is_weekend'] = (df['observation_date'].dt.dayofweek >= 5).astype(int)
    # df['is weekend'] = np.where(df['observation_date'].dt.dayofweek>=5,1,0)

    df['hour_sin'] = np.sin(2*np.pi*df['hour']/24)
    df['hour_cos'] = np.cos(2*np.pi*df['hour']/24)

    return df
    raise NotImplementedError('Complete load_data')


In [ ]:
def build_pipeline(algorithm: BaseEstimator) -> Pipeline:
    """Build an unfitted pipeline with training-only numeric/category preparation."""

    num_pipeline = Pipeline([
        ('imputer',SimpleImputer(strategy='median')),
        ('scaler',StandardScaler())
    ])

    preprocessor = ColumnTransformer([
        ('numeric',num_pipeline,NUMERIC_FEATURES),
        ('category',OneHotEncoder(handle_unknown='ignore'),CATEGORICAL)
    ])

    return Pipeline([
        ('preprocessor',preprocessor),('model',algorithm)
    ])
    raise NotImplementedError('Complete build_pipeline')


In [ ]:
def prepare_datasets(train_file: str, test_file: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """Return clean predictor/target pairs for chronological train/test files."""

    train_df = load_data(train_file)
    test_df = load_data(test_file)

    for df in train_df,test_df:
        df[TARGET] = pd.to_numeric(df[TARGET],errors='coerce').replace(['',np.inf,-np.inf],np.nan)
        df.dropna(subset=[TARGET],inplace=True)
        df.drop(df[df[TARGET]<0].index,inplace=True)

    X_train = train_df[FEATURES]
    X_test = test_df[FEATURES]
    y_train = train_df[TARGET]
    y_test = test_df[TARGET]

    return X_train,X_test,y_train,y_test
    raise NotImplementedError('Complete prepare_datasets')


In [ ]:
def train_models(X_train: pd.DataFrame, y_train: pd.Series) -> dict[str, Pipeline]:
    """Fit two separate complete demand-estimation pipelines."""

    models = {
        'decision_tree' : DecisionTreeRegressor(max_depth=10, min_samples_leaf=4, random_state=42),
        'random_forest' : RandomForestRegressor(n_estimators=120, max_depth=14, min_samples_leaf=2, random_state=42, n_jobs=-1)
    }
    
    trained_models = {}
    for name,model in models.items():
        pipeline = build_pipeline(model)
        pipeline.fit(X_train,y_train)
        trained_models[name] = pipeline
    
    return trained_models
    raise NotImplementedError('Complete train_models')


In [ ]:
def evaluate_models(trained_models: dict[str, Pipeline], X_test: pd.DataFrame, y_test: pd.Series) -> pd.DataFrame:
    """Score fitted pipelines and rank by lowest RMSLE."""

    results = []

    for name,model in trained_models.items():
        predictions = np.maximum(model.predict(X_test),0)
        results.append({
            'model_name' : name,
            'mae' : mean_absolute_error(y_test,predictions),
            'rmse' : np.sqrt(mean_squared_error(y_test,predictions)),
            'rmsle' : np.sqrt(mean_squared_log_error(y_test,predictions))
        })

    comparison = pd.DataFrame(results).sort_values(['rmsle']).reset_index(drop=True)

    return comparison
    raise NotImplementedError('Complete evaluate_models')


In [ ]:
def select_best_model(comparison: pd.DataFrame, trained_models: dict[str, Pipeline]) -> tuple[str, Pipeline]:
    """Select the first model from the RMSLE-ranked comparison."""

    best_name = comparison.iloc[0]['model_name']
    best_model = trained_models[best_name]

    return best_name,best_model
    raise NotImplementedError('Complete select_best_model')


In [ ]:
def predict_demand(model: Pipeline, df: pd.DataFrame) -> pd.DataFrame:
    """Return nonnegative whole-session demand predictions for queued rows."""

    predictions = np.rint(np.maximum(model.predict(df[FEATURES]),0)).astype(int)

    return pd.DataFrame({
        'observation_id' : df['observation_id'].to_numpy(),
        'predicted_sessions' : predictions
    })
    
    raise NotImplementedError('Complete predict_demand')


## 3. Independent training and prediction workflows

Training saves the fitted pipeline and comparison. Prediction loads it and never calls `fit`.

In [ ]:
def train() -> tuple[str, pd.DataFrame]:
    """Train, compare and save the best fitted pipeline and report."""

    X_train,X_test,y_train,y_test = prepare_datasets('ev_train.csv','ev_test.csv')

    trained_models = train_models(X_train,y_train)

    comparison = evaluate_models(trained_models,X_test,y_test)

    best_name,best_model = select_best_model(comparison,trained_models)

    joblib.dump(best_model,'best_model.pkl')
    comparison.to_csv('model_comparison.csv',index=False)

    return best_name,comparison
    raise NotImplementedError('Complete train')


In [ ]:
def predict() -> pd.DataFrame:
    """Load the saved pipeline and estimate queued sessions without retraining."""

    data = load_data('ev_predict.csv')
    model = joblib.load('best_model.pkl')

    predictions = predict_demand(model,data)

    predictions.to_csv('ev_predictions.csv',index=False)

    return predictions
    raise NotImplementedError('Complete predict')


## 4. Run the workflow

Execute the **training** call first, then the **prediction** call.

In [ ]:
# Uncomment to train after completing the functions:
# best_name, comparison = train()
# display(comparison)


In [ ]:
# Uncomment to predict after training:
# ev_predictions = predict()
# display(ev_predictions.head())
